# SPF Current-Quarter Baseline — GDP Nowcast Comparison

Computes the Survey of Professional Forecasters (SPF) current-quarter GDP nowcast from the nominal (NGDP) level series and compares it against the GNN–Pearson model on the evaluation window.

In [ ]:
import pandas as pd, numpy as np, pathlib

GN_DIR  = pathlib.Path("./data/gdpnow_output")
RES_CSV = pathlib.Path("./data/output/new/"
                       "GNN_corr/quarterly_walkfwd_v72_final/walkforward_results.csv")

# SPF median NOMINAL GDP (NGDP) level -> in-vintage current-quarter SAAR
#   NGDP1 = panel's contemporaneous prior-quarter level (t-1)
#   NGDP2 = panel's median forecast level for the current quarter (t)
spf = pd.read_excel(GN_DIR / "spf_median_ngdp_level.xlsx")
spf["date"] = pd.to_datetime(
    spf["YEAR"].astype(int).astype(str) + "-" +
    spf["QUARTER"].astype(int).map({1:"01", 2:"04", 3:"07", 4:"10"}) + "-01")
spf["spf_saar"] = ((spf["NGDP2"] / spf["NGDP1"]) ** 4 - 1) * 100
spf = spf.set_index("date")[["spf_saar"]].dropna()
print(f"SPF rows: {len(spf)}  |  coverage: {spf.index.min().date()} -> {spf.index.max().date()}")

In [ ]:
# GNN-Pearson results (actual_saar = final-revised nominal SAAR target)
gnn = pd.read_csv(RES_CSV)
gnn["date"] = pd.to_datetime(gnn["test_quarter"]); gnn = gnn.set_index("date")

common = spf.index.intersection(gnn.index)
m = gnn.loc[common].join(spf)
m["spf_ae"] = (m["spf_saar"] - m["actual_saar"]).abs()

print("="*60)
print("SPF CURRENT-QUARTER  vs  GNN-PEARSON   (nominal, in-vintage)")
print("="*60)
print(f"  Overlap quarters : {len(m)}")
print(f"  SPF  MAE         : {m['spf_ae'].mean():.3f} pp")
print(f"  SPF  RMSE        : {np.sqrt((m['spf_ae']**2).mean()):.3f} pp")
print(f"  GNN  MAE         : {m['saar_abs_error'].mean():.3f} pp")
print(f"  GNN wins         : {(m['saar_abs_error'] < m['spf_ae']).sum()}/{len(m)} quarters")

In [ ]:
# Era-by-era MAE: SPF vs GNN
eras = [("Pre-GFC",2006,2007),("GFC",2008,2009),("Recovery",2010,2014),
        ("Expansion",2015,2019),("COVID",2020,2021),("Post-COVID",2022,2025)]
rows = []
for name,lo,hi in eras:
    s = m[(m.index.year>=lo)&(m.index.year<=hi)]
    if len(s):
        rows.append({"Era":name,"N":len(s),
                     "SPF MAE":round(s["spf_ae"].mean(),2),
                     "GNN MAE":round(s["saar_abs_error"].mean(),2)})
rows.append({"Era":"Full overlap","N":len(m),
             "SPF MAE":round(m["spf_ae"].mean(),2),
             "GNN MAE":round(m["saar_abs_error"].mean(),2)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# NBER recession-quarter AE: SPF vs GNN
#   GFC negative quarters (Q1, Q4 2008; Q1, Q2 2009) + COVID recession (Q1, Q2 2020)
rec = pd.to_datetime([f"{y}-{mo:02d}-01" for y,mo in
       [(2008,1),(2008,10),(2009,1),(2009,4),(2020,1),(2020,4)]])
rr = []
for q in rec:
    if q in m.index:
        r = m.loc[q]
        rr.append({"Quarter":f"{q.year}-Q{(q.month-1)//3+1}",
                   "Actual SAAR":round(r["actual_saar"],1),
                   "SPF Pred":round(r["spf_saar"],1),
                   "SPF AE":round(r["spf_ae"],2),
                   "GNN AE":round(r["saar_abs_error"],2)})
rdf = pd.DataFrame(rr)
print(rdf.to_string(index=False))
print(f"\nMean SPF AE (6 recession qtrs): {rdf['SPF AE'].mean():.2f} pp | mean GNN AE: {rdf['GNN AE'].mean():.2f} pp")

## Notes

The SPF current-quarter nowcast is derived from the nominal NGDP level series and evaluated against the same final-revised GDP target as the model.